<a href="https://colab.research.google.com/github/VictorLemosFr/Desafio-Dados-PTC-2026.1/blob/main/Treinamento_PTC0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🟢 Treinamento de Dados — CITi PTC 26.1

Bem-vindo(a)! Esse notebook é um guia rápido para você entender o básico do que vai usar no desafio.

Vamos cobrir 3 coisas:
1. **Explorar** uma base de dados com Pandas
2. **Limpar** as inconsistências mais comuns
3. **Entender** como funciona um agente de IA simples

---
## 📦 Parte 1 — Carregando e explorando os dados

Primeiro, vamos importar o Pandas (a biblioteca que usamos para trabalhar com tabelas em Python) e carregar o nosso mini dataset.


In [ ]:
import pandas as pd

# Carrega o CSV — substitua pelo caminho do seu arquivo
df = pd.read_csv('mini_base_financeira_suja.csv')

# Ver as primeiras linhas
df.head()

,ID_Transacao,Banco,Nome_Cliente,Data_Transacao,Valor_Transacao,Tipo_Transacao,Status_Transacao,Categoria,Num_Parcelas,Taxa_Servico,Valor_Final
0,TXN001,Nubank,Ana Silva,2024-01-15,"R$ 1.500,00",PIX,Aprovada,Alimentação,1,15.0,1515.00
1,TXN002,Bradesco,BRUNO OLIVEIRA,15/01/2024,320.50,pgto.,APROVADA,Saúde,3,5.0,325.50
2,TXN003,Itaú,carlos souza,01-15-2024,BRL 750.00,Transferência,Recusada,Transporte,1,10.0,760.00
3,TXN004,Nubank,Ana Silva,2024-01-15,"R$ 1.500,00",PIX,Aprovada,Alimentação,1,15.0,1515.00
4,TXN005,Santander,maria.santos@,January 20 2024,9999999.99,TRANSFERÊNCIA,aprov.,Lazer,2,20.0,9999999.99


In [ ]:
# Quantas linhas e colunas?
print('Shape:', df.shape)

# Tipos de dados e valores nulos
df.info()

Shape: (33, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID_Transacao      31 non-null     object 
 1    Banco            31 non-null     object 
 2   Nome_Cliente      29 non-null     object 
 3   Data_Transacao    31 non-null     object 
 4   Valor_Transacao   31 non-null     object 
 5   Tipo_Transacao    30 non-null     object 
 6   Status_Transacao  30 non-null     object 
 7   Categoria         29 non-null     object 
 8   Num_Parcelas      31 non-null     object 
 9   Taxa_Servico      29 non-null     float64
 10  Valor_Final       31 non-null     float64
dtypes: float64(2), object(9)
memory usage: 3.0+ KB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Quantos nulos por coluna?
df.isnull().sum()

,0
ID_Transacao,2
Banco,2
Nome_Cliente,4
Data_Transacao,2
Valor_Transacao,2
Tipo_Transacao,3
Status_Transacao,3
Categoria,4
Num_Parcelas,2
Taxa_Servico,4


---
## 🧹 Parte 2 — Limpeza de dados

Aqui estão as operações mais importantes que você vai precisar no desafio.

In [ ]:
# ✅ 1. Limpar espaços nos nomes das colunas
df.columns = df.columns.str.strip()
print(df.columns.tolist())

['ID_Transacao', 'Banco', 'Nome_Cliente', 'Data_Transacao', 'Valor_Transacao', 'Tipo_Transacao', 'Status_Transacao', 'Categoria', 'Num_Parcelas', 'Taxa_Servico', 'Valor_Final']


In [ ]:
# ✅ 2. Remover linhas completamente vazias e duplicatas
df = df.dropna(how='all')       # remove linhas onde TUDO é nulo
df = df.drop_duplicates()        # remove linhas idênticas

print('Linhas restantes:', len(df))

Linhas restantes: 31


In [ ]:
# ✅ 3. Padronizar nomes (formato Título, sem espaços extras)
df['Nome_Cliente'] = df['Nome_Cliente'].str.strip()          # remove espaços
df['Nome_Cliente'] = df['Nome_Cliente'].str.replace(r'[^a-zA-ZÀ-ÿ\s]', '', regex=True)  # remove caracteres especiais
df['Nome_Cliente'] = df['Nome_Cliente'].str.title()          # Formato Título

df['Nome_Cliente'].unique()

array(['Ana Silva', 'Bruno Oliveira', 'Carlos Souza', 'Mariasantos',
       'Fernanda Lima', 'João Pereira', nan, 'Fernandalima'], dtype=object)

In [ ]:
# ✅ 4. Limpar e converter Valor_Transacao para número
df['Valor_Transacao'] = (
    df['Valor_Transacao']
    .astype(str)
    .str.replace('R$', '', regex=False)
    .str.replace('BRL', '', regex=False)
    .str.replace('.', '', regex=False)   # remove separador de milhar
    .str.replace(',', '.', regex=False)  # transforma vírgula em ponto decimal
    .str.strip()
)
df['Valor_Transacao'] = pd.to_numeric(df['Valor_Transacao'], errors='coerce')

# Remove outliers absurdos (negativos, zero, acima de 100.000)
df = df[df['Valor_Transacao'].between(0.01, 100000)]

# Preenche nulos com a mediana
mediana = df['Valor_Transacao'].median()
df['Valor_Transacao'] = df['Valor_Transacao'].fillna(mediana)

df['Valor_Transacao'].describe()

,Valor_Transacao
count,26.000000
mean,28842.384615
std,28925.187567
min,1.000000
25%,1500.000000
50%,22275.000000
75%,49750.000000
max,89000.000000


In [ ]:
# ✅ 5. Padronizar Tipo_Transacao
mapa_tipo = {
    'transferencia': 'Transferência',
    'transferência': 'Transferência',
    'transf.': 'Transferência',
    'pgto.': 'Pagamento',
    'pagamento': 'Pagamento',
    'pix': 'PIX',
    'p.i.x': 'PIX',
    'pix': 'PIX',
    'pixe': 'PIX',
    'depósito': 'Depósito',
    'deposito': 'Depósito',
    'saque': 'Saque',
    'retirada': 'Saque',
}

df['Tipo_Transacao'] = (
    df['Tipo_Transacao']
    .str.strip()
    .str.lower()
    .map(mapa_tipo)
    .fillna(df['Tipo_Transacao'].mode()[0])  # preenche desconhecidos com a moda
)

df['Tipo_Transacao'].value_counts()

,count
Tipo_Transacao,
PIX,9
Pagamento,5
Saque,5
Transferência,4
Depósito,3


In [ ]:
# ✅ 6. Padronizar Status_Transacao
mapa_status = {
    'aprovada': 'Aprovada',
    'aprovado': 'Aprovada',
    'aprov.': 'Aprovada',
    'recusada': 'Recusada',
    'negada': 'Recusada',
    'pendente': 'Pendente',
    'pend.': 'Pendente',
    'em análise': 'Pendente',
}

df['Status_Transacao'] = (
    df['Status_Transacao']
    .str.strip()
    .str.lower()
    .map(mapa_status)
)

df['Status_Transacao'].value_counts()

,count
Status_Transacao,
Aprovada,15
Recusada,5
Pendente,5


In [ ]:
# ✅ 7. Ver o resultado final
df.head(10)

,ID_Transacao,Banco,Nome_Cliente,Data_Transacao,Valor_Transacao,Tipo_Transacao,Status_Transacao,Categoria,Num_Parcelas,Taxa_Servico,Valor_Final
0,TXN001,Nubank,Ana Silva,2024-01-15,1500.0,PIX,Aprovada,Alimentação,1,15.0,1515.000
1,TXN002,Bradesco,Bruno Oliveira,15/01/2024,32050.0,Pagamento,Aprovada,Saúde,3,5.0,325.500
2,TXN003,Itaú,Carlos Souza,01-15-2024,75000.0,Transferência,Recusada,Transporte,1,10.0,760.000
3,TXN004,Nubank,Ana Silva,2024-01-15,1500.0,PIX,Aprovada,Alimentação,1,15.0,1515.000
5,TXN006,Bradesco,Fernanda Lima,2024/01/22,43000.0,Saque,NaN,Outros,1,-8.0,422.000
6,TXN007,Nubank,João Pereira,22.01.2024,1.0,PIX,Pendente,Alimentação,1,5.0,5.001
7,TXN008,Caixa,NaN,2024-01-25,50000.0,Depósito,Aprovada,Saúde,única,12.0,512.000
9,TXN010,Santander,Fernanda Lima,2024-01-28,1200.0,Transferência,Pendente,Lazer,3 parcelas,NaN,1225.000
10,TXN011,Bradesco,Carlos Souza,28.01.2024,89000.0,Saque,Recusada,NaN,1,10.0,850.000
11,TXN012,Nubank,João Pereira,2024/01/30,2300.0,Transferência,Aprovada,Alimentação,6.0,30.0,2330.000


In [ ]:
# Salvar o CSV limpo
df.to_csv('base_financeira_limpa.csv', index=False)
print('✅ Base limpa salva!')

✅ Base limpa salva!


In [1]:
!pip install google-genai

In [ ]:
from google import genai

# ── Dados ──────────────────────────────────────────────────────────────────────

resumo = """
Resumo:
- Total de projetos vendidos: 1000
- Valor total movimentado: R$ 100.000.000
- Os produtos mais vendidos: agentes de IA, automações, chatbots
- Melhor empresa do mundo: CITi
- A melhor subárea do CITi é dados
"""

# ── Cliente ────────────────────────────────────────────────────────────────────

client = genai.Client(api_key="SUA CHAVE DA API")

# ── Loop principal ─────────────────────────────────────────────────────────────

pergunta = ""

while pergunta != "FIM":
    pergunta = input("Qual é a sua pergunta? ")

    if pergunta == "FIM":
        continue

    prompt = (
        f"{resumo}\n\n"
        f"Com base nesses dados, sempre responda qualquer tipo de pergunta baseado "
        f"nos dados que você recebeu, não importa a pergunta. "
        f"Se você não recebeu nenhuma informação sobre isso, fale 'Não tenho essa informação': "
        f"{pergunta}"
    )

    resposta = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )

    print("\n🤖 Resposta da IA:")
    print(resposta.text)